## 1. Import Libraries

In [ ]:
import os
import glob
import datetime
import statistics
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

## 2. Load & Clean ES Data

**If running via Google Colab, you first need to upload the data to the session drive:**  
1. Click the folder icon on the left sidebar
2. Click the file icon (Upload to session storage)
3. Locate the folder of .csv files (`.../Box/EventPerception/keptdata_20views`)
4. Highlight all .csv files (shift+click) and upload

The script will outpute the number of .csv files found.

In [ ]:


datapath = '/Users/justincampbell/Desktop/EventSeg/'
savepath = '/Users/justincampbell/Desktop/EventSeg/Results'
files = glob.glob(os.path.join(datapath, '*Event_Perception*.csv'))
print('Found %i files' %(len(files)))

The function `ESFileCleaner()` loads in the raw .csv files and organizes the data into a Pandas DataFrame.

In [ ]:
def ESFileCleaner(filepath):
    # this function prepares the raw .csv file for plotting / analysis

    try:
      # load data
      df = pd.read_csv(filepath)

      # pull key info
      pID = df['participant'].unique()
      videos = df['movie_filename'].unique()
      RT1 = df['resp1.rt'].unique()
      RT2 = df['resp2.rt'].unique()
      RT3 = df['resp3.rt'].unique()
      Age = df['age'].unique()
      Sex = df['sex'].unique()


      # remove empty rows
      pID = pID[~pd.isnull(pID)][0]
      videos = videos[~pd.isnull(videos)]
      RT1 = RT1[~pd.isnull(RT1)][0]
      RT2 = RT2[~pd.isnull(RT2)][0]
      RT3 = RT3[~pd.isnull(RT3)][0]
      Age = Age[~pd.isnull(Age)][0]
      Sex = Sex[~pd.isnull(Sex)][0]

      # clean-up video names
      videos = [x.split('/')[1] for x in videos]
      videos = [x.split('.')[0] for x in videos]

      # parse video details
      pIDNum = pID[2:]
      pIDNum = pIDNum.replace('O','0')
      pIDNum = int(pIDNum)
      RWNum = [x.split('_')[0][-1] for x in videos]
      walkNum = [x.split('_')[1] for x in videos]

      # clean-up RTs
      def RTCleaner(RT):
          # this function fixes formatting issues from loading the .csv file
          RT = RT.replace('[','')
          RT = RT.replace(']','')
          RT = RT.replace(' ', '')
          RT = RT.split(',')
          RT = np.array(RT)
          RT = [float(x) for x in RT]
          return RT

      RT1 = RTCleaner(RT1)
      RT2 = RTCleaner(RT2)
      RT3 = RTCleaner(RT3)

      # export to table
      TABLElen = len([videos[0]] * len(RT1) + [videos[1]] * len(RT2) + [videos[2]] * len(RT3))

      ESTable = pd.DataFrame(data = {'pID': ([pID] * TABLElen),
                                  'pIDNum': ([pIDNum] * TABLElen),
                                  'Video': ([videos[0]] * len(RT1) + [videos[1]] * len(RT2) + [videos[2]] * len(RT3)),
                                  'RWN': ([RWNum[0]] * len(RT1) + [RWNum[1]] * len(RT2) + [RWNum[2]] * len(RT3)),
                                  'Walk': ([walkNum[0]] * len(RT1) + [walkNum[1]] * len(RT2) + [walkNum[2]] * len(RT3)),
                                  'Event': np.concatenate([RT1,RT2,RT3])})

      # fix variable types
      ESTable['pIDNum'] = pd.to_numeric(ESTable['pIDNum'])
      ESTable['RWN'] = pd.to_numeric(ESTable['RWN'])
      ESTable['Walk'] = pd.to_numeric(ESTable['Walk'])
      ESTable['Event'] = pd.to_numeric(ESTable['Event'])

      # reset index
      ESTable = ESTable.reset_index(drop=True)

    except:
      path, file = os.path.split(filepath)
      print('Could not process %s...' %file)
      return

    return ESTable

In [ ]:
ESTable = pd.concat([ESFileCleaner(file) for file in files])

In [ ]:
# Export
ESTable.to_csv('EventSegTable.csv')